# 4 - Mesh and density

Two builders turn scattered points into a continuous surface: `trimesh` (a Delaunay triangulation coloured
by a value - an interpolated field from samples) and `kde` (a kernel-density estimate - a smooth heat map of
where points concentrate).

**Setup** and data - the Heidelberg bird observations.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # deliberately silence all warnings to keep the rendered docs clean

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import geopandas as gpd
import numpy as np

birds = gpd.read_file(str(DATA / "heidelberg_birds_2023.geojson"))
birds["log_uncertainty"] = np.log10(birds["uncertainty_m"].clip(lower=1.0))
print("birds:", len(birds))

## `trimesh` - an interpolated field from samples

`trimesh` triangulates the points and colours each face by `value_column`, turning sparse samples into a
continuous surface. `rasterize=True` shades the triangulation for a dense cloud.

In [ ]:
m = InteractiveMap(crs=4326, title="bird location uncertainty (trimesh)")
m.trimesh(birds, value_column="log_uncertainty", rasterize=True, cmap="plasma")
show(m)

## `kde` - kernel density

`kde` estimates a smooth density surface from the points - a filled heat map of concentration.

In [ ]:
m = InteractiveMap(crs=4326, title="bird observation density (kde)")
m.kde(birds, filled=True, cmap="magma")
show(m)